# Test the model published on Hugging Face

Downloads **your model from the Hugging Face Hub** (not from Drive), checks that the upload is complete, generates Stories from the downloaded files, and (optionally) proves that the Hub copy is the **same model** you trained on Drive. It does **no training**. Run the cells top to bottom.

- Put your repo in `HF_REPO_ID` (cell 1). It is left blank in the committed file on purpose.
- A **private** repo needs a Colab secret named `HF_TOKEN` (key icon in the left sidebar, a token with **read** access, "Notebook access" switched on). A public repo needs no token.
- A GPU is not required (the model is small), but **Runtime -> Change runtime type -> GPU** makes it faster.
- Cell 1 has `QUIT_WHEN_DONE = True`: the last cell ends the session so no credits are spent while you read. Set it to `False` to keep experimenting.
- Nothing is uploaded or changed on the Hub by this notebook.

In [ ]:
# 1. Settings + Google Drive (Drive is only used for the optional checks in cells 6 and 10 and to keep the GIFs)
HF_REPO_ID = ""           # the model repo to test, e.g. "your-username/dlm-tinystories-36m_cosine" (type yours here)
SCHEDULE = "cosine"       # the noise schedule the model was trained with ("linear" or "cosine"); generation uses the same one
COMPARE_WITH_DRIVE = True # cell 6: check the Hub weights are identical to the trained model on Drive, and that greedy text matches
RUN_NAME = "pretrain_cosine"  # the Drive run folder under checkpoints/ that cell 6 compares with
RUN_EVAL = True           # cell 10: measure likelihood + sample statistics of the Hub model (needs the token cache on Drive; ~5-10 min)
QUIT_WHEN_DONE = True     # the LAST cell ends the Colab session (nothing needs saving: the model lives on the Hub)
GRACE_SECONDS = 60        # countdown before the session ends (stop the last cell during it to cancel)

from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/diffusion_lm_outputs"
RUN = f"{OUT}/checkpoints/{RUN_NAME}"
LOCAL = f"{RUN}/final"    # the Drive copy of the trained model (only used by cell 6)

In [ ]:
# 2. Get the latest code from GitHub (public repo; tokenizer/ is committed there). Re-running this cell is safe.
REPO = "https://github.com/bitlabsdevteam/Diffusion-Language-Model.git"
import os
if os.path.isdir("/content/dlm_repo"):
    !git -C /content/dlm_repo pull --ff-only
else:
    !git clone -q $REPO /content/dlm_repo
%cd /content/dlm_repo
!git log --oneline -1

In [ ]:
# 3. Dependencies: install ONLY what is missing (never numpy / torch: reinstalling them can break Colab's live kernel)
import importlib.util, subprocess, sys
missing = [p for p in ("tokenizers", "safetensors", "huggingface_hub", "pyarrow", "datasets") if importlib.util.find_spec(p) is None]
print("missing:", missing or "none")
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def sh(*args):
    """Run `python run.py <args>` and print its output."""
    p = subprocess.run([sys.executable, "run.py", *args], capture_output=True, text=True)
    print(p.stdout + p.stderr)
    return p.returncode

In [ ]:
# 4. Download the model from the Hub and check the upload is complete. CKPT = the downloaded folder: every later cell uses ONLY these
# files (model.safetensors, config.json, tokenizer.json), exactly what a stranger who downloads your repo would get.
import json, os
if not HF_REPO_ID:
    raise SystemExit("Set HF_REPO_ID in cell 1 (for example 'your-username/dlm-tinystories-36m_cosine'), then run cells 1 and 4 again.")
from dlm.hub import HubError, download_snapshot
try:
    CKPT = download_snapshot(HF_REPO_ID)
except HubError as e:
    raise SystemExit(f"{e}\nA private repo needs a Colab secret named HF_TOKEN (read access, 'Notebook access' on). "
                     "Also check the repo name for typos.")
print("Downloaded to", CKPT, "\n")
files = sorted((os.path.relpath(os.path.join(d, f), CKPT), os.path.getsize(os.path.join(d, f)))
               for d, _, fs in os.walk(CKPT) for f in fs if not d.split(os.sep)[-1].startswith("."))
for name, size in files:
    print(f"  {name:32s} {size / 1e6:8.2f} MB")
names = {n for n, _ in files}
need = ("config.json", "model.safetensors", "tokenizer.json", "generation_config.json", "README.md")
absent = [n for n in need if n not in names]
print("\nAll required files are on the Hub." if not absent else f"\nMISSING from the Hub repo: {absent}")
if os.path.isfile(f"{CKPT}/run_info.json"):
    info = json.load(open(f"{CKPT}/run_info.json"))
    r = info.get("recipe", {})
    print(f"Trained for {r.get('steps')} steps, noise schedule {r.get('schedule')!r}; stopped_at_step = {info.get('stopped_at_step')}")
    if r.get("schedule") and r["schedule"] != SCHEDULE:
        print(f"WARNING: the model was trained with the {r['schedule']!r} schedule but SCHEDULE = {SCHEDULE!r} in cell 1")
if os.path.isfile(f"{CKPT}/README.md") and "intermediate checkpoint" in open(f"{CKPT}/README.md").read().lower():
    print("NOTE: the model card says this is an intermediate checkpoint (it was exported from resume.pt before the run ended).")
gen_cfg = json.load(open(f"{CKPT}/generation_config.json")) if "generation_config.json" in names else {}
if gen_cfg.get("schedule") not in (None, SCHEDULE):
    print(f"NOTE: the repo's generation_config.json says schedule = {gen_cfg['schedule']!r}; the sampler here ignores that file "
          f"and uses SCHEDULE = {SCHEDULE!r} from cell 1.")

In [ ]:
# 5. What the Hub page shows: the training curves and the evaluation report that were uploaded with the model.
from IPython.display import Image, display
if os.path.isfile(f"{CKPT}/training_curves.png"):
    display(Image(f"{CKPT}/training_curves.png"))
else:
    print("No training_curves.png in the Hub repo.")
if os.path.isfile(f"{CKPT}/eval/report.md"):
    print(open(f"{CKPT}/eval/report.md").read())
else:
    print("No eval/report.md in the Hub repo.")

In [ ]:
# 6. Is the Hub copy the SAME model as the one trained on Drive? Compares every weight, then greedy text (temperature 0 is deterministic,
# so equal weights must give equal text). Skipped when COMPARE_WITH_DRIVE = False or there is no final/ on Drive.
if not COMPARE_WITH_DRIVE:
    print("Skipped (COMPARE_WITH_DRIVE = False).")
elif not os.path.isfile(f"{LOCAL}/model.pt"):
    print(f"Skipped: no trained model at {LOCAL}/model.pt (make it with: python run.py export --resume {RUN}/resume.pt --out {LOCAL}).")
else:
    import torch
    from dlm.sample import generate_text, load_checkpoint
    hub_model, hub_tok = load_checkpoint(CKPT, "cpu")
    drive_model, drive_tok = load_checkpoint(LOCAL, "cpu")
    a, b = hub_model.state_dict(), drive_model.state_dict()
    same_keys = a.keys() == b.keys()
    worst = max((a[k].float() - b[k].float()).abs().max().item() for k in a) if same_keys else float("nan")
    print(f"weights: same tensor names = {same_keys}; largest difference = {worst:g}")
    print("tokenizers identical:", hub_tok.to_str() == drive_tok.to_str())
    good = same_keys and worst == 0.0
    for prompt in ("", "Once upon a time, there was a little"):
        kw = dict(prompt=prompt, seed=0, max_new_tokens=96, temperature=0.0, schedule=SCHEDULE)
        t_hub = generate_text(hub_model, hub_tok, **kw)["text"]
        t_drive = generate_text(drive_model, drive_tok, **kw)["text"]
        print(f"greedy text from {'a prompt' if prompt else 'scratch'}: {'IDENTICAL' if t_hub == t_drive else 'DIFFERENT'}")
        good = good and t_hub == t_drive
    print("\nRESULT:", "the Hub model is identical to the trained model on Drive." if good else "MISMATCH: the Hub model differs from the trained model, do not trust the upload.")

In [ ]:
# 7. Three Stories from scratch (no prompt), generated from the downloaded Hub files. Each line below the Story is its length and whether it finished.
sh("generate", "--ckpt", CKPT, "--n", "3", "--seed", "0", "--schedule", SCHEDULE)

In [ ]:
# 8. Continue YOUR opening words (edit the form and re-run as often as you like). --trace shows the unmasking as text rows
# (every row = one step; blocks of 32 are filled left to right, most confident first). Avoid double quotes in the prompt.
PROMPT = "Once upon a time, there was a little"  #@param {type:"string"}
HOW_MANY = 2  #@param {type:"integer"}
SEED = 0  #@param {type:"integer"}
TEMPERATURE = 0.8  #@param {type:"number"}
import glob, os
from IPython.display import Image, display
for f in glob.glob("/content/unmask*.gif"):
    os.remove(f)
sh("generate", "--ckpt", CKPT, "--prompt", PROMPT, "--n", str(HOW_MANY), "--seed", str(SEED),
   "--temperature", str(TEMPERATURE), "--schedule", SCHEDULE, "--trace", "--gif", "/content/unmask.gif")
gifs = sorted(glob.glob("/content/unmask*.gif"))
if gifs:
    display(Image(gifs[0]))   # the unmasking animation of the first sample

In [ ]:
# 9. Does decoding speed or randomness change quality? Same opening words: greedy (temperature 0, deterministic), the default,
# and a fast setting (60 steps for ~480 tokens = about 8 tokens per step). Compare the text and the printed time.
import time
for label, extra in (("greedy", ["--temperature", "0"]), ("default", []), ("fast, 60 steps", ["--steps", "60"])):
    print(f"\n===== {label} =====")
    t0 = time.time()
    sh("generate", "--ckpt", CKPT, "--prompt", PROMPT, "--n", "1", "--seed", "1", "--schedule", SCHEDULE, *extra)
    print(f"[{label}: {time.time() - t0:.1f} s including start-up]")

In [ ]:
# 10. Measure the Hub model (optional, RUN_EVAL): validation likelihood next to the 5.92 token-frequency baseline, and statistics of
# generated Stories vs real ones. A QUICK version on a subset of the validation blocks. Needs the token cache on Drive (made by the
# data cells of run_colab.ipynb). Results go to /content/hf_eval (the Hub download folder is left untouched).
if not RUN_EVAL:
    print("Skipped (RUN_EVAL = False).")
elif not os.path.isfile(f"{OUT}/data/tokens/tokens_manifest.json"):
    print(f"Skipped: no token cache at {OUT}/data/tokens (run the data cells of run_colab.ipynb once, or set RUN_EVAL = False).")
else:
    !mkdir -p /content/tokens && cp -r $OUT/data/tokens/. /content/tokens/
    !python run.py eval --ckpt $CKPT --cache /content/tokens --out /content/hf_eval --n-stories 50 --n-prompts 10 --draws 2 --max-val-blocks 1500 --no-gpt2
    print(open("/content/hf_eval/report.md").read())
    print(open("/content/hf_eval/samples.md").read()[:5000])

In [ ]:
# 11. END THE SESSION. Nothing needs saving (the model is on the Hub); any unmask*.gif is copied to Drive first.
try:
    from dlm.colab import finish_session
    finish_session(OUT, run=RUN_NAME, enabled=QUIT_WHEN_DONE, grace_seconds=GRACE_SECONDS, require_model=False, require_eval=False)
except Exception as e:  # never end the session by accident
    print(f"NOT ending the session ({type(e).__name__}: {e}). Stop it yourself: Runtime -> Disconnect and delete runtime.")